<a href="https://colab.research.google.com/github/Sbolivar16/MolecularDocking/blob/main/UniGBSA_PostDocking_Colab_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uni-GBSA — MM/GBSA and MM/PBSA as a post-docking filter
### Google Colab workflow for a receptor + docking pose(s)

This notebook uses **Uni-GBSA** as a post-docking rescoring step.

## What should the user upload?

**1. Receptor:** a protein in `.pdb` format.

**2. Ligand pose(s):** one or more structures obtained from docking. Accepted formats are:

- `.sdf`
- `.mol`
- `.mol2`
- `.pdb`
- AutoDock Vina/Gnina `.pdbqt`

PDBQT files are automatically converted to SDF while attempting to preserve the Cartesian coordinates of the docking pose.

> Uni-GBSA does not require the user to upload a single combined protein–ligand PDB file. The official `unigbsa-pipeline` workflow uses the receptor and ligand as separate inputs.

## Recommended workflow

**Docking → selected pose → Uni-GBSA (EM + MM/GBSA) → ranking → extended MD → MM/PBSA/MM/GBSA on the trajectory**

For an initial post-docking filter, use:

- `mode = em`
- `modes = gb`

This minimizes the pose before the MM/GBSA calculation.


## 0. Install Uni-GBSA

A separate Conda environment with Python 3.11 is created to avoid compatibility issues with the Python version provided by Google Colab.


In [ ]:
import os, subprocess

MINIFORGE = "/content/miniforge3"
CONDA = f"{MINIFORGE}/bin/conda"
ENV = "gbsa"

if not os.path.exists(CONDA):
    subprocess.run([
        "bash","-lc",
        "wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh "
        "-O /content/miniforge.sh && bash /content/miniforge.sh -b -p /content/miniforge3"
    ], check=True)

install_cmd = f'''
source {MINIFORGE}/etc/profile.d/conda.sh
conda config --set channel_priority strict
if ! conda env list | awk '{{print $1}}' | grep -qx {ENV}; then
    conda create -y -n {ENV} -c conda-forge python=3.11 acpype openmpi mpi4py gromacs "gmx_mmpbsa>=1.5.6" openbabel
fi
conda run -n {ENV} pip install -q unigbsa lickit
'''
subprocess.run(["bash","-lc",install_cmd], check=True)

print("✅ Uni-GBSA environment installed.")


In [ ]:
import subprocess, os

MINIFORGE = "/content/miniforge3"
ENV = "gbsa"

def env_run(command, check=True, capture=False):
    full = f"source {MINIFORGE}/etc/profile.d/conda.sh && conda run -n {ENV} bash -lc {command!r}"
    return subprocess.run(["bash","-lc",full], check=check, text=True, capture_output=capture)

checks = [
    ("Uni-GBSA", "unigbsa-pipeline -h | head -n 3"),
    ("GROMACS", "gmx --version | head -n 2"),
    ("ACPYPE", "acpype -h | head -n 2"),
    ("gmx_MMPBSA", "gmx_MMPBSA -h | head -n 2"),
    ("Open Babel", "obabel -V"),
]

for name, cmd in checks:
    print(f"--- {name} ---")
    r = env_run(cmd, check=False, capture=True)
    print(((r.stdout or r.stderr) or "No output")[:1200])
    print()

print("✅ Verification completed.")


## 1. Create working directories


In [ ]:
import os, glob, shutil
from google.colab import files

ROOT = "/content/UniGBSA_PostDocking"
RECEPTOR_DIR = os.path.join(ROOT, "receptor")
LIGAND_IN_DIR = os.path.join(ROOT, "input_ligands")
LIGAND_SDF_DIR = os.path.join(ROOT, "ligand_sdf")
RESULT_DIR = os.path.join(ROOT, "results")

for d in [RECEPTOR_DIR, LIGAND_IN_DIR, LIGAND_SDF_DIR, RESULT_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Working directories created.")


## 2. Upload the receptor PDB


In [ ]:
print("Select the receptor in PDB format:")
uploaded = files.upload()

RECEPTOR_PDB = None
for name, data in uploaded.items():
    if name.lower().endswith(".pdb"):
        RECEPTOR_PDB = os.path.join(RECEPTOR_DIR, "receptor.pdb")
        with open(RECEPTOR_PDB, "wb") as f:
            f.write(data)
        break

if RECEPTOR_PDB is None:
    raise ValueError("No .pdb file was uploaded.")

print("✅ Receptor:", RECEPTOR_PDB)


## 3. Upload one or more docking poses


In [ ]:
print("Select one or more docking poses:")
uploaded = files.upload()

allowed = {".pdbqt", ".sdf", ".mol2", ".mol", ".pdb"}
ligand_inputs = []

for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    if ext in allowed:
        out = os.path.join(LIGAND_IN_DIR, os.path.basename(name))
        with open(out, "wb") as f:
            f.write(data)
        ligand_inputs.append(out)
    else:
        print("⚠️ Ignored:", name)

if not ligand_inputs:
    raise ValueError("No compatible ligand file was uploaded.")

print(f"✅ {len(ligand_inputs)} ligand(s) uploaded.")
for p in ligand_inputs:
    print(" -", os.path.basename(p))


## 4. Convert poses to SDF

Uni-GBSA works with ligands in MOL/SDF format. If a pose comes from Vina/Gnina in PDBQT format, it is converted with Open Babel **without using `--gen3d`**, so the docking geometry is preserved.

Pay special attention to molecular connectivity if the ligand contains non-standard chemistry, metals, or groups with ambiguous bond orders.


In [ ]:
def convert_to_sdf(path):
    base = os.path.splitext(os.path.basename(path))[0]
    out = os.path.join(LIGAND_SDF_DIR, base + ".sdf")
    ext = os.path.splitext(path)[1].lower()

    if ext == ".sdf":
        shutil.copy2(path, out)
        return out

    r = env_run(f'obabel "{path}" -O "{out}"', check=False, capture=True)

    if r.returncode != 0 or not os.path.exists(out) or os.path.getsize(out) == 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(f"Conversion failed for {path}")
    return out

sdf_files = []

for lig in ligand_inputs:
    try:
        sdf = convert_to_sdf(lig)
        sdf_files.append(sdf)
        print("✅", os.path.basename(lig), "→", os.path.basename(sdf))
    except Exception as e:
        print("❌", e)

if not sdf_files:
    raise RuntimeError("No SDF file could be generated.")


## 5. Visual inspection


In [ ]:
!pip -q install py3Dmol > /dev/null 2>&1
import py3Dmol

with open(RECEPTOR_PDB, "r", errors="ignore") as f:
    receptor_text = f.read()

with open(sdf_files[0], "r", errors="ignore") as f:
    ligand_text = f.read()

view = py3Dmol.view(width=750, height=500)
view.addModel(receptor_text, "pdb")
view.setStyle({"model":0}, {"cartoon":{"color":"spectrum"}})
view.addModel(ligand_text, "sdf")
view.setStyle({"model":1}, {"stick":{"radius":0.22}})
view.zoomTo()
view.show()

print("Check that the pose remains located in the correct binding site.")


## 6. Calculation parameters

For a fast post-docking filter, the recommended starting setup is:

- **Mode:** `em`
- **Energy model:** `gb`
- **Ligand:** GAFF + AM1-BCC
- **IGB:** 2
- **Interior dielectric:** 4.0
- **Exterior dielectric:** 80.0

`input` mode uses the docking pose directly.  
`em` mode performs energy minimization.  
`md` mode runs a short automated molecular dynamics simulation.


## Uni-GBSA calculation setup

This section defines the parameters that control how Uni-GBSA prepares the protein–ligand complex and which model is used to estimate the binding energy.

Uni-GBSA can be used as a **post-docking rescoring** step, i.e., as an additional filter after AutoDock Vina to prioritize complexes that will later be subjected to more extensive molecular dynamics simulations.

The selected setup should be kept constant for all compounds that will be compared.

---

## 1. Calculation mode (`Mode`)

The **Mode** parameter determines which structural treatment is applied to the complex before the energy calculation.

### `input`

Uses the protein–ligand conformation provided by the user directly, without prior energy minimization.

The workflow is:

**Docking → input structure → MM/GBSA or MM/PBSA**

#### When to use it

- when you want to evaluate the docking pose exactly as obtained;
- for quick tests;
- when you want to compare the energy before and after minimization.

#### Limitation

A docking pose may contain locally unfavorable contacts or small steric clashes. Therefore, the result may depend strongly on the initial geometry.

For post-docking screening, this is generally not the preferred option.

---

### `em` — Energy Minimization

Performs **energy minimization of the protein–ligand complex** before calculating MM/GBSA or MM/PBSA.

The workflow is:

**Docking → energy minimization → MM/GBSA or MM/PBSA**

Minimization can relax:

- steric contacts;
- local orientations;
- ligand geometry;
- protein–ligand interactions.

#### When to use it

This is the recommended option when Uni-GBSA is used as a **post-docking filter**.

Example workflow:

**Docking of multiple compounds**  
↓  
**Uni-GBSA (`em + GB`)**  
↓  
**selection of the best candidates**  
↓  
**extended molecular dynamics**

This option provides a practical balance between computational cost and structural refinement.

#### Recommended post-docking setup

`Mode = em`

---

### `md` — Molecular Dynamics

Runs a short molecular dynamics simulation before the energy calculation.

The workflow is:

**Docking → system preparation → short MD → structure extraction → MM/GBSA or MM/PBSA**

#### When to use it

- for a reduced number of candidates;
- when dynamic relaxation of the complex is desired;
- as an intermediate step before a longer molecular dynamics simulation.

#### Consideration

It is considerably more expensive than `input` or `em`, so it is usually not convenient for hundreds or thousands of ligands.

For an initial post-docking screen, use `em` and reserve `md` for the highest-ranked candidates.

---

## 2. Energy model (`Energy`)

Uni-GBSA can use two main implicit-solvation models:

### `GB` — Generalized Born

Calculates the energy using **MM/GBSA**.

In simplified form:

$$
\Delta G_{\mathrm{bind}} =
\Delta E_{\mathrm{MM}} +
\Delta G_{\mathrm{solv}} -
T\Delta S
$$

where:

- $\Delta G_{\mathrm{bind}}$ is the estimated binding free energy;
- $\Delta E_{\mathrm{MM}}$ is the molecular mechanics contribution;
- $\Delta G_{\mathrm{solv}}$ is the solvation contribution;
- $T\Delta S$ is the entropic term.

In the GB model, the polar solvation contribution is estimated using the **Generalized Born** approach.

#### Advantages

- relatively fast calculation;
- suitable for comparing many complexes;
- widely used as a rescoring method;
- lower computational cost than PB.

#### Recommended use

For the initial post-docking filter:

`Energy = GB`

---

### `PB` — Poisson–Boltzmann

Calculates the energy using **MM/PBSA**.

In this case, the polar solvation contribution is obtained by solving the **Poisson–Boltzmann** equation, which usually requires more computation than GB.

#### Advantages

- can provide a more detailed electrostatic description;
- useful for complementing GB results;
- appropriate for analyzing a smaller number of candidates.

#### When to use it

- for more detailed analyses;
- for energetic refinement of the best candidates;
- to complement an initial GB-based filter.

For initial screening, it is usually reasonable to start with:

`Energy = GB`

---

## 3. Protein force field (`Protein FF`)

This parameter defines the set of parameters used to describe the protein atoms.

For example:

`amber03`

corresponds to the **AMBER03** force field.

The force field defines parameters such as:

- bond lengths;
- angles;
- torsions;
- partial charges;
- Lennard-Jones parameters.

### Recommendation

All complexes in the same study should use **exactly the same protein force field**.

Do not calculate one ligand with one protein force field and another ligand with a different one if their energies will later be compared.

For comparative screening, methodological consistency is essential.

---

## 4. Ligand force field (`Ligand FF`)

This parameter controls how organic molecules are parameterized.

### `gaff`

Corresponds to the **General AMBER Force Field**.

It is one of the most widely used force fields for parameterizing small organic molecules and is designed to work together with AMBER-family protein force fields.

### `gaff2`

A newer version of GAFF with modifications to several torsional and non-bonded parameters.

### Recommendation

If you use:

`Ligand FF = gaff`

all ligands in the study should be prepared with the same scheme.

Do not mix GAFF and GAFF2 within the same energetic ranking.

---

## 5. Charge assignment method (`Charge`)

Ligand partial charges are especially important because they directly affect:

- electrostatic interactions;
- hydrogen bonds;
- solvation energy;
- MM/GBSA or MM/PBSA energy.

### `bcc`

Usually corresponds to the **AM1-BCC** charge scheme.

AM1-BCC uses a semiempirical approach followed by bond charge corrections.

It is widely used together with GAFF because it provides a practical balance between speed and charge quality.

#### Recommended use

For small organic molecules:

`Charge = bcc`

is a reasonable choice for screening studies.

---

### `gas`

This option uses charges obtained through the procedure configured for gas-phase charge calculations.

Its exact meaning depends on the implementation used internally by Uni-GBSA.

To maintain a standardized protocol for small molecules with GAFF, the usual recommendation is:

`Charge = bcc`

---

## 6. Number of threads (`Threads`)

This parameter defines the number of CPU threads used during the calculation.

Example:

`Threads = 2`

means that Uni-GBSA can use two processing threads.

Increasing the number of threads may accelerate some stages, although the speedup is not always linear.

### In Google Colab

A conservative setup is:

`Threads = 2`

This avoids requesting more resources than the environment normally provides.

If the same workflow is run on a workstation or cluster, the number of available threads can be increased.

---

## 7. Per-residue decomposition (`Per-residue decomposition`)

This option performs a **decomposition of the binding energy by protein residue**.

Instead of obtaining only a global energy:

$$
\Delta G_{\mathrm{bind}}
$$

the calculation also estimates how much each protein residue contributes to the interaction with the ligand.

A conceptual result could look like this:

| Residue | Energy contribution |
|---|---:|
| PHE57 | −2.8 kcal/mol |
| ASP120 | −2.1 kcal/mol |
| LEU69 | −1.7 kcal/mol |
| ARG58 | −1.4 kcal/mol |

More negative values are generally interpreted as more favorable contributions within the selected model.

### When to enable it

It is especially useful for:

- identifying important residues in the binding site;
- comparing residues that stabilize different ligands;
- relating energetic results to interactions observed during docking;
- supporting mechanistic interpretation;
- generating tables or figures for publications.

### When to leave it disabled

If the goal is only to rank many compounds quickly, it can be left disabled because it increases the calculation time and the amount of output generated.

A practical strategy is:

**First filter:**  
`Decomposition = OFF`

**Final-candidate analysis:**  
`Decomposition = ON`

---

# Recommended setup for a post-docking filter

To use Uni-GBSA immediately after AutoDock Vina, a reasonable initial setup is:

| Parameter | Recommended setting |
|---|---|
| Mode | `em` |
| Energy | `GB` |
| Protein FF | `amber03` |
| Ligand FF | `gaff` |
| Charge | `bcc` |
| Threads | `2` |
| Per-residue decomposition | Disabled |

The workflow would be:

**AutoDock Vina**  
↓  
**pose selection**  
↓  
**Uni-GBSA**  
`em + MM/GBSA`  
↓  
**energy ranking**  
↓  
**candidate selection**  
↓  
**extended molecular dynamics**  
↓  
**MM/PBSA or MM/GBSA over multiple trajectory frames**

---

## Important interpretation notes

The energy calculated by Uni-GBSA from a minimized pose should be considered primarily a **rescoring and prioritization** tool.

It should not be interpreted as an absolute experimental free energy.

In addition:

- the AutoDock Vina score and the MM/GBSA $\Delta G$ are not directly equivalent;
- a small difference between two molecules does not necessarily imply a biologically meaningful difference;
- all ligands should be analyzed using exactly the same protocol;
- results should be interpreted comparatively within the same dataset;
- final candidates should be evaluated using longer molecular dynamics simulations and energy analysis over multiple trajectory structures.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

mode_widget = widgets.Dropdown(options=["em","input","md"], value="em", description="Mode:")
energy_widget = widgets.SelectMultiple(options=["gb","pb"], value=("gb",), description="Energy:")
protein_ff_widget = widgets.Dropdown(options=["amber03","amber99sb","amber99sb-ildn"], value="amber03", description="Protein FF:")
ligand_ff_widget = widgets.Dropdown(options=["gaff","gaff2"], value="gaff", description="Ligand FF:")
charge_widget = widgets.Dropdown(options=["bcc","gas"], value="bcc", description="Charge:")
threads_widget = widgets.IntSlider(value=2, min=1, max=4, step=1, description="Threads:")
decomp_widget = widgets.Checkbox(value=False, description="Per-residue decomposition")

display(mode_widget, energy_widget, protein_ff_widget, ligand_ff_widget, charge_widget, threads_widget, decomp_widget)


## 7. Generate the `config.ini` file


In [ ]:
CONFIG = os.path.join(ROOT, "unigbsa_postdocking.ini")
modes = ",".join(energy_widget.value)

lines = [
    "[simulation]",
    f"mode = {mode_widget.value}",
    "boxtype = triclinic",
    "boxsize = 0.9",
    "conc = 0.15",
    "nsteps = 500000",
    "eqsteps = 50000",
    "nframe = 100",
    f"proteinforcefield = {protein_ff_widget.value}",
    f"ligandforcefield = {ligand_ff_widget.value}",
    f"ligandCharge = {charge_widget.value}",
    "",
    "[GBSA]",
    "sys_name = PostDocking",
    f"modes = {modes}",
    "igb = 2",
    "indi = 4.0",
    "exdi = 80.0",
]

with open(CONFIG, "w") as f:
    f.write("\n".join(lines) + "\n")

print(open(CONFIG).read())


## 8. Run Uni-GBSA


In [ ]:
OUTCSV = os.path.join(RESULT_DIR, "BindingEnergy.csv")
decomp_flag = "--decomp" if decomp_widget.value else ""

cmd = (
    f'unigbsa-pipeline '
    f'-i "{RECEPTOR_PDB}" '
    f'-d "{LIGAND_SDF_DIR}" '
    f'-c "{CONFIG}" '
    f'-o "{OUTCSV}" '
    f'-nt {threads_widget.value} '
    f'{decomp_flag} --verbose'
)

print("Command:")
print(cmd)
print("\nRunning...\n")

r = env_run(cmd, check=False, capture=False)

if r.returncode == 0:
    print("\n✅ Uni-GBSA completed successfully.")
    print("Output:", OUTCSV)
else:
    print("\n❌ Uni-GBSA finished with an error.")
    print("Check the previous ACPYPE/GROMACS/gmx_MMPBSA messages.")


## 9. Results table and ranking


In [ ]:
import pandas as pd, glob, os

if os.path.exists(OUTCSV):
    df = pd.read_csv(OUTCSV)
    display(df)

    candidates = [c for c in df.columns if any(k in c.lower() for k in ["delta", "binding", "total"])]
    if candidates:
        col = candidates[0]
        try:
            ranking = df.sort_values(col, ascending=True).reset_index(drop=True)
            ranking.insert(0, "Rank", range(1, len(ranking)+1))
            display(ranking)
            ranking.to_csv(os.path.join(RESULT_DIR, "Ranking_UniGBSA.csv"), index=False)
        except Exception:
            pass
else:
    print("BindingEnergy.csv was not found.")


# 📊 How to interpret Uni-GBSA results

The table generated by Uni-GBSA summarizes the main energetic contributions involved in estimating protein–ligand binding energy.

In general, the estimated binding energy can be written as:

$$
\Delta G_{\mathrm{bind}}
\approx
\Delta E_{\mathrm{MM}}
+
\Delta G_{\mathrm{solv}}
$$

where:

$$
\Delta E_{\mathrm{MM}}
=
\Delta E_{\mathrm{vdW}}
+
\Delta E_{\mathrm{elec}}
+
\Delta E_{\mathrm{internal}}
$$

and:

$$
\Delta G_{\mathrm{solv}}
=
\Delta G_{\mathrm{polar}}
+
\Delta G_{\mathrm{nonpolar}}
$$

At this **post-docking rescoring** stage, the most important column for building a comparative ranking is:

`TOTAL`

However, the final value is much more informative when interpreted together with its individual contributions.

---

# 🧪 Worked example

For the complex:

`etq_v2_vina`

Uni-GBSA produced approximately the following values:

| Component | Energy (kcal/mol) |
|---|---:|
| Internal | ≈ 0.00 |
| Van der Waals | −48.95 |
| Electrostatic | −3.52 |
| Polar Solvation | +9.15 |
| Non-Polar Solvation | −6.08 |
| Gas | −52.47 |
| Solvation | +3.06 |
| **TOTAL** | **−49.40** |

The final result is:

`TOTAL = −49.40 kcal/mol`

This negative value indicates that, within the selected energy model, complex formation is energetically favorable.

---

# 🧩 1. `Internal`: internal energy

In this example:

`Internal ≈ 0 kcal/mol`

Internal energy includes contributions mainly associated with:

- bonds;
- angles;
- torsions;
- intramolecular terms.

In a single-trajectory scheme, or in calculations where receptor, ligand, and complex share the same reference geometry, these contributions may cancel almost completely.

Therefore, in this case:

`Internal ≈ 0`

is not a problem.

It simply indicates that this component contributes very little to the final binding value.

---

# 🧲 2. `Van der Waals`: molecular contacts and complementarity

In the example:

`Van der Waals = −48.95 kcal/mol`

This is the largest favorable contribution in the system.

Van der Waals interactions mainly represent short-range contacts between ligand atoms and atoms in the binding pocket.

They include:

- dispersion forces;
- geometric complementarity;
- contacts between molecular surfaces;
- favorable interactions between non-polar regions.

A negative value means that these interactions favor complex formation.

## Reading this example

`−48.95 kcal/mol`

is a strongly favorable contribution.

This suggests that the ligand has good spatial complementarity with the binding pocket.

In this complex, van der Waals interactions are clearly the main favorable energetic driver.

---

# ⚡ 3. `Electrostatic`: direct electrostatic interaction

In the example:

`Electrostatic = −3.52 kcal/mol`

This component represents direct electrostatic interactions between partial charges on the ligand and the protein.

It may include contributions associated with:

- charged groups;
- dipoles;
- hydrogen bonds;
- ion–dipole interactions;
- interactions between polar regions.

A negative value represents a favorable contribution.

## Reading this example

`−3.52 kcal/mol`

indicates that electrostatic interactions favor binding, although their magnitude is considerably smaller than the van der Waals contribution.

Therefore, the calculated stabilization of this system appears to be dominated mainly by van der Waals contacts, while direct electrostatics plays a secondary role.

---

# 💧 4. `Polar Solvation`: electrostatic solvation cost

In the example:

`Polar Solvation = +9.15 kcal/mol`

This positive value represents an unfavorable contribution to binding.

However, it is important to understand what this term actually means.

## Does Uni-GBSA remove water molecules?

**No.**

In MM/GBSA, the polar-solvation component is calculated using an **implicit solvent** model.

This means that water is not represented during this energy calculation as thousands of individual molecules surrounding the system.

Instead, the solvent is treated as a **continuous medium** with defined dielectric properties.

For MM/GBSA, the polar contribution is estimated with the **Generalized Born** model.

Conceptually:

$$
\Delta G_{\mathrm{polar}}
=
G_{\mathrm{polar}}^{complex}
-
G_{\mathrm{polar}}^{receptor}
-
G_{\mathrm{polar}}^{ligand}
$$

In other words, the algorithm compares the electrostatic solvation energy of the complex with the corresponding energies of the separated receptor and ligand.

---

## What does “polar desolvation” mean?

When the protein and ligand are separated, many of their polar or charged groups are favorably stabilized by the aqueous environment.

For example, they can interact with the solvent through:

- electrostatic interactions;
- hydrogen bonds;
- dielectric stabilization.

When the complex forms, part of those polar surfaces may become buried and less exposed to solvent.

The implicit-solvent model estimates the energetic effect of that change.

If the new protein–ligand interactions do not fully compensate for the loss of solvent-mediated electrostatic stabilization, a positive penalty appears.

In simplified terms, this is called:

**polar desolvation penalty**

Importantly:

> Uni-GBSA does not identify or physically remove water molecules one by one.

The penalty arises from the energetic difference calculated using a continuum-solvent model.

---

## What if `Mode = md` is used?

If Uni-GBSA is run in:

`md`

mode, system preparation or molecular simulation may involve an explicit water box during the dynamics stage.

In that context, water molecules may indeed be represented explicitly during the simulation.

However, when MM/GBSA is subsequently calculated, the solvation energy is generally evaluated with the corresponding implicit-solvent model.

Therefore, two concepts should be distinguished:

### Explicit-solvent molecular dynamics

**Protein + ligand + water molecules**

### MM/GBSA

**System coordinates + continuum-solvent model**

---

## Reading this example

In this system:

`Polar Solvation = +9.15 kcal/mol`

means that the polar-solvation component penalizes complex formation by approximately:

**+9.15 kcal/mol**

It does not mean that the algorithm removed polar groups or individual water molecules.

It means that the electrostatic model calculates a polar-solvation cost for complex formation relative to the separated receptor and ligand.

---

# 🌊 5. `Non-Polar Solvation`: burial of non-polar surface

In the example:

`Non-Polar Solvation = −6.08 kcal/mol`

The negative value represents a favorable contribution.

A more precise way to describe this component is:

**burial of non-polar surface**

rather than simply “hydrophobic burial.”

---

## What does this contribution represent?

The non-polar solvation component is mainly related to changes in solvent-accessible surface area:

**SASA — Solvent Accessible Surface Area**

In simplified form:

$$
G_{\mathrm{nonpolar}}
\approx
\gamma \cdot SASA + \beta
$$

where:

- $SASA$ is the solvent-accessible surface area;
- $\gamma$ is a parameter related to surface tension;
- $\beta$ is an additional model-dependent term.

When the ligand enters the binding pocket, part of the non-polar surface of both the ligand and protein becomes less exposed to solvent.

This process can produce a favorable change in the non-polar solvation component.

---

## What can this component reflect?

It can be associated with:

- burial of non-polar surface;
- reduction of solvent-exposed area;
- formation of a protein–ligand interface that is less exposed to water;
- contributions related to cavitation and dispersion, depending on the model.

Therefore, this component should not be interpreted as a direct measure of the “hydrophobic effect,” but rather as an energetic estimate mainly related to changes in non-polar solvent-accessible surface.

---

## Reading this example

In this system:

`Non-Polar Solvation = −6.08 kcal/mol`

indicates that burial of non-polar surface during complex formation contributes approximately:

**−6.08 kcal/mol**

favorably.

This contribution helps compensate for part of the polar penalty described above.

---

# 🌐 6. `Gas`: direct protein–ligand interactions

The:

`Gas`

column represents the sum of the main direct molecular interaction terms.

In simplified form:

$$
\Delta G_{\mathrm{gas}}
\approx
\Delta E_{\mathrm{vdW}}
+
\Delta E_{\mathrm{elec}}
+
\Delta E_{\mathrm{internal}}
$$

For this example:

$$
\Delta G_{\mathrm{gas}}
\approx
-48.95
-
3.52
+
0
$$

$$
\Delta G_{\mathrm{gas}}
\approx
-52.47\ \mathrm{kcal/mol}
$$

which matches:

`Gas = −52.47 kcal/mol`

---

## Interpretation

The strongly negative value indicates that direct protein–ligand interactions are energetically favorable.

In this example, the favorable value is dominated mainly by:

`Van der Waals = −48.95 kcal/mol`

---

# 💦 7. `Solvation`: net solvation effect

The:

`Solvation`

column is approximately:

$$
\Delta G_{\mathrm{solv}}
=
\Delta G_{\mathrm{polar}}
+
\Delta G_{\mathrm{nonpolar}}
$$

For this example:

$$
\Delta G_{\mathrm{solv}}
=
9.15
-
6.08
$$

$$
\Delta G_{\mathrm{solv}}
\approx
+3.07\ \mathrm{kcal/mol}
$$

The reported value is approximately:

`Solvation = +3.06 kcal/mol`

---

## Interpretation

The polar component penalizes binding:

`+9.15 kcal/mol`

while the non-polar component favors binding:

`−6.08 kcal/mol`

The final balance remains slightly unfavorable:

`+3.06 kcal/mol`

This means that, overall, solvation partially reduces the favorability of direct protein–ligand interactions.

---

# ⭐ 8. `TOTAL`: the main value for ranking

The:

`TOTAL`

column is the main result used to compare ligands.

It is approximately:

$$
\Delta G_{\mathrm{TOTAL}}
=
\Delta G_{\mathrm{gas}}
+
\Delta G_{\mathrm{solv}}
$$

For this example:

$$
\Delta G_{\mathrm{TOTAL}}
=
-52.47
+
3.06
$$

$$
\Delta G_{\mathrm{TOTAL}}
\approx
-49.40\ \mathrm{kcal/mol}
$$

The resulting value is:

`TOTAL = −49.40 kcal/mol`

---

## What does it mean?

A negative value indicates that, within the selected model, complex formation is energetically favorable.

When comparing different ligands calculated with exactly the same protocol:

**more negative values → more favorable calculated interaction**

For example:

| Ligand | TOTAL (kcal/mol) |
|---|---:|
| Ligand A | −55.3 |
| Ligand B | −49.4 |
| Ligand C | −37.8 |
| Ligand D | −21.5 |

The energetic priority would be:

**Ligand A > Ligand B > Ligand C > Ligand D**

---

# 🧠 Integrated interpretation of `etq_v2_vina`

The complete result can be read as follows:

### ✅ Very favorable direct interactions

`Gas = −52.47 kcal/mol`

mainly because of:

`Van der Waals = −48.95 kcal/mol`

This suggests good spatial complementarity between the ligand and the pocket.

---

### ✅ Favorable electrostatics

`Electrostatic = −3.52 kcal/mol`

Electrostatics contributes favorably, although less strongly than van der Waals interactions.

---

### ⚠️ Polar penalty

`Polar Solvation = +9.15 kcal/mol`

There is an electrostatic solvation cost associated with complex formation.

---

### ✅ Non-polar compensation

`Non-Polar Solvation = −6.08 kcal/mol`

Burial of non-polar surface compensates for an important part of the polar cost.

---

### ⚖️ Solvation balance

`Solvation = +3.06 kcal/mol`

Overall solvation is slightly unfavorable.

---

### ⭐ Final result

`TOTAL = −49.40 kcal/mol`

The favorable direct interactions clearly outweigh the solvation penalty.

Therefore, within this protocol, the pose:

`etq_v2_vina`

shows a favorable calculated interaction.

---

# ⚠️ Important: `TOTAL` is not equivalent to the AutoDock Vina score

Suppose Vina produced:

`Vina = −8.2 kcal/mol`

and Uni-GBSA produced:

`MM/GBSA = −49.4 kcal/mol`

This does not mean that Uni-GBSA predicted an interaction that is six times stronger.

The two values come from different energy models.

The correct workflow is:

**AutoDock Vina**

↓  

**pose selection**

↓  

**Uni-GBSA / MM-GBSA**

↓  

**energy rescoring**

↓  

**candidate selection**

---

# ⏱️ Important: `Frames = 1`

In this example:

`Frames = 1`

This means that the energy was obtained from a single structure.

If the calculation was performed with:

`Mode = em`

that structure essentially corresponds to the complex relaxed by energy minimization.

Therefore:

`TOTAL = −49.40 kcal/mol`

should be interpreted as:

**MM/GBSA rescoring of a minimized pose**

and not as a converged free-energy estimate derived from an extended molecular dynamics trajectory.

---

# 🔬 What should be done with the best candidates?

After the Uni-GBSA filter, the best candidates can proceed to:

**100–200 ns molecular dynamics**

↓  

**structural analysis**

- RMSD
- RMSF
- radius of gyration
- SASA
- hydrogen bonds
- PCA
- FEL

↓  

**MM/PBSA or MM/GBSA over multiple frames**

↓  

**mean energy + standard deviation + standard error**

This provides a much more robust characterization than evaluating a single pose.

---

# 🎯 Which columns should you inspect first?

For a quick interpretation:

### 1️⃣ `TOTAL`
Main value for energy-based ranking.

### 2️⃣ `Van der Waals`
Reports favorable contact/complementarity contributions.

### 3️⃣ `Electrostatic`
Reports the direct electrostatic contribution.

### 4️⃣ `Polar Solvation`
Reports the electrostatic cost associated with the change in solvation.

### 5️⃣ `Non-Polar Solvation`
Mainly reflects burial of non-polar surface.

### 6️⃣ `Gas`
Summarizes the direct molecular interaction terms.

### 7️⃣ `Solvation`
Reports the net solvation balance.

---

# 🧾 Example summary

| Component | Value | Interpretation |
|---|---:|---|
| Internal | ≈ 0.00 | Nearly cancels |
| Van der Waals | −48.95 | Very favorable |
| Electrostatic | −3.52 | Favorable |
| Polar Solvation | +9.15 | Unfavorable |
| Non-Polar Solvation | −6.08 | Favorable |
| Gas | −52.47 | Very favorable direct interactions |
| Solvation | +3.06 | Slightly unfavorable |
| **TOTAL** | **−49.40** | **Favorable overall balance** |

---

## 📌 Final message

The `TOTAL` value should be used primarily to **compare and prioritize molecules calculated under exactly the same conditions**.

It should not be interpreted in isolation or as an absolute experimental affinity.

The main role of Uni-GBSA in this workflow is to act as an intermediate filter:

**Docking → Uni-GBSA → selection → molecular dynamics → MM/PBSA/MM-GBSA over the trajectory**


## 10. Download all results


In [ ]:
import zipfile, os
from google.colab import files

ZIP_OUT = "/content/UniGBSA_PostDocking_Results.zip"

with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fs in os.walk(ROOT):
        for fname in fs:
            path = os.path.join(root, fname)
            z.write(path, arcname=os.path.relpath(path, ROOT))

print("✅", ZIP_OUT)
files.download(ZIP_OUT)


# Interpretation

For a screening workflow:

**Docking → Uni-GBSA EM/GB → prioritization → extended MD**

A more negative `ΔG` represents a more favorable calculated interaction **within the same Uni-GBSA protocol**.

It should not be interpreted as equivalent to the Vina score or as an experimental free energy.

For the final candidates, a more robust analysis is:

**Explicit-solvent MD → multiple frames → MM/PBSA or MM/GBSA over the trajectory**
